# E1 V3 — Parameter Optimization (Optuna)

**Strategy:** V3_Short_EMA200 — longs always, shorts only when close < EMA(ema_period)

**LOCKED BOUNDARIES:**
- Train (optimize here): 2023-01-01 → 2024-06-30
- Validation (locked reference): 2024-07-01 → 2025-01-01 — do NOT run until params are final
- Out-of-sample: 2025-01-01 → today — DO NOT TOUCH

**Params to tune (4 only):**
1. atr_compression_threshold
2. volume_floor_multiplier (longs and shorts share same floor)
3. range_period
4. short_trend_ema_period

**Longs logic: unchanged.** Shorts: EMA-gated only.

**Objective:** composite score (not raw Sharpe) — see cell below.

In [ ]:
import sys, warnings, datetime, traceback
warnings.filterwarnings("ignore")
sys.path.insert(0, '/Users/hermes/quants-lab')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from decimal import Decimal
from core.backtesting import BacktestingEngine
from core.backtesting.optimizer import BacktestingConfig, BaseStrategyConfigGenerator, StrategyOptimizer
from app.controllers.directional_trading.e1_compression_breakout import E1CompressionBreakoutConfig
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
from hummingbot.core.data_type.common import OrderType, TradeType
import numpy as np

# ── HARD-CODED TRAIN WINDOW ──────────────────────────────────────────────────
TRAIN_START = datetime.datetime(2023, 1, 1)
TRAIN_END   = datetime.datetime(2024, 6, 30)
# ─────────────────────────────────────────────────────────────────────────────

# Minimum trades required — avoid over-optimizing on sparse samples
MIN_TRADES = 15

print(f"Train window: {TRAIN_START.date()} → {TRAIN_END.date()}")
print("V3 variant: longs always, shorts gated by close < EMA(ema_period)")

## Composite Objective

Raw Sharpe is noisy at small sample sizes and doesn't capture regime robustness.

Composite score:
- Base: Sharpe ratio (primary)
- Bonus: +0.5 if profit_factor > 2.0
- Bonus: +0.3 if win_rate > 0.60
- Penalty: -1.0 if max_drawdown_pct < -0.08 (>8% DD on $1k)
- Penalty: -2.0 if n_trades < MIN_TRADES (sparse sample — not trustworthy)
- Hard floor: return -inf if net_pnl_quote <= 0 (unprofitable runs disqualified)

In [ ]:
def composite_score(results, executors) -> float:
    """
    Composite objective for V3 optimization.
    Prioritizes: bull robustness, controlled DD, meaningful sample size.
    NOT raw Sharpe or raw PnL.
    """
    n = results.get("total_positions", 0)
    net_pnl = results.get("net_pnl_quote", 0)
    sharpe = results.get("sharpe_ratio", 0) or 0
    pf = results.get("profit_factor", 0) or 0
    accuracy = results.get("accuracy", 0) or 0
    max_dd_pct = results.get("max_drawdown_pct", 0) or 0

    # Hard disqualifiers
    if net_pnl <= 0:
        return float('-inf')
    if n < MIN_TRADES:
        return -2.0  # not enough trades to trust

    score = sharpe

    # Quality bonuses
    if pf > 2.0:
        score += 0.5
    if accuracy > 0.60:
        score += 0.3

    # DD penalty
    if max_dd_pct < -0.08:
        score -= 1.0

    # Sparse penalty (graduated)
    if n < 25:
        score -= 0.3

    return float(score)


print("Composite objective defined")
print(f"Min trades threshold: {MIN_TRADES}")

In [ ]:
class E1V3ConfigGenerator(BaseStrategyConfigGenerator):
    """
    V3 config generator — 4 params only.
    Longs: unchanged. Shorts: EMA-gated (close < EMA(short_trend_ema_period)).
    Train window hard-coded — validation window never touched.
    """

    async def generate_config(self, trial) -> BacktestingConfig:

        # ── 4 params to tune ────────────────────────────────────────────────
        atr_threshold = trial.suggest_float(
            "atr_compression_threshold", 0.10, 0.35, step=0.05)
        vol_floor = trial.suggest_float(
            "volume_floor_multiplier", 1.1, 2.0, step=0.1)
        range_period = trial.suggest_int(
            "range_period", 10, 40, step=5)
        ema_period = trial.suggest_int(
            "short_trend_ema_period", 50, 300, step=50)

        # ── Locked params ────────────────────────────────────────────────────
        config = E1CompressionBreakoutConfig(
            id=f"e1_v3_opt_{trial.number}",
            connector_name="binance_perpetual",
            trading_pair="BTC-USDT",
            total_amount_quote=Decimal("1000"),
            max_executors_per_side=1,
            cooldown_time=300,
            leverage=1,
            atr_period=14,                        # locked
            atr_compression_window=90,            # locked (days)
            atr_compression_threshold=atr_threshold,
            range_period=range_period,
            volume_period=20,                     # locked
            volume_floor_multiplier=vol_floor,
            volume_spike_multiplier=2.0,          # locked
            risk_off_ema_period=20,               # locked
            risk_off_threshold=0.97,              # locked
            # V3: longs always, shorts gated by EMA
            trade_direction="BOTH",
            short_trend_filter=True,
            short_ema_slope_filter=False,         # locked off for V3
            short_trend_ema_period=ema_period,
            short_volume_floor_multiplier=vol_floor,  # same floor for both sides
            stop_loss=Decimal("0.015"),           # locked
            take_profit=Decimal("0.03"),          # locked
            time_limit=86400,                     # locked
            trailing_stop=TrailingStop(
                activation_price=Decimal("0.015"),
                trailing_delta=Decimal("0.005"),
            ),
            take_profit_order_type=OrderType.MARKET,
        )

        return BacktestingConfig(
            config=config,
            start=self.start,
            end=self.end,
        )


print("E1V3ConfigGenerator ready")
print(f"Search space: atr_threshold [0.10-0.35], vol_floor [1.1-2.0], range_period [10-40], ema_period [50-300]")

## Run Optimization on Train Window Only

In [ ]:
# Fresh engine per optimizer (avoids state corruption between runs)
class E1V3Optimizer(StrategyOptimizer):
    """Override _async_objective to use composite score instead of raw Sharpe."""

    async def _async_objective(self, trial, config_generator) -> float:
        try:
            bt_config = await config_generator.generate_config(trial)
            # Fresh engine per trial to avoid state corruption
            from core.backtesting import BacktestingEngine
            bt = BacktestingEngine(load_cached_data=True)
            result = await bt.run_backtesting(
                config=bt_config.config,
                start=bt_config.start,
                end=bt_config.end,
                backtesting_resolution="1m",
                trade_cost=0.000375,
            )
            r = result.results
            if not isinstance(r.get("close_types"), dict):
                r["close_types"] = {}

            # Log key attrs for analysis
            for key in ["net_pnl_quote","sharpe_ratio","profit_factor","accuracy",
                        "max_drawdown_pct","total_positions","total_long","total_short"]:
                trial.set_user_attr(key, r.get(key, 0))

            score = composite_score(r, result.executors)
            print(f"  Trial {trial.number}: score={score:.3f} sharpe={r.get('sharpe_ratio',0):.3f} "
                  f"pnl=${r.get('net_pnl_quote',0):.1f} n={r.get('total_positions',0)} "
                  f"atr={trial.params.get('atr_compression_threshold')} "
                  f"vol={trial.params.get('volume_floor_multiplier')} "
                  f"rng={trial.params.get('range_period')} "
                  f"ema={trial.params.get('short_trend_ema_period')}")
            return score

        except Exception as e:
            print(f"  Trial {trial.number} FAILED: {e}")
            traceback.print_exc()
            return float('-inf')


config_generator = E1V3ConfigGenerator(start_date=TRAIN_START, end_date=TRAIN_END)
optimizer = E1V3Optimizer(load_cached_data=True)

print(f"Starting V3 optimization — train window {TRAIN_START.date()} → {TRAIN_END.date()}")
print("Objective: composite score (Sharpe + quality bonuses - DD penalty)")
print("Each trial creates a fresh BacktestingEngine to avoid state corruption\n")

await optimizer.optimize(
    study_name="e1_v3_short_ema_gated",
    config_generator=config_generator,
    n_trials=100,
    load_if_exists=True,
)
print("\nOptimization complete")

## Reload study (run this if kernel was interrupted)

In [ ]:
import optuna
storage = "sqlite:////Users/hermes/quants-lab/app/data/processed/backtesting/optimization_database.db"
study = optuna.load_study(study_name="e1_v3_short_ema_gated", storage=storage)
optimizer.study = study
complete = [t for t in study.trials if t.state.name == "COMPLETE"]
print(f"Trials complete: {len(complete)} / {len(study.trials)}")
if complete:
    print(f"Best composite score: {study.best_value:.4f}")
    print(f"Best params: {study.best_params}")

## Analyze Results

In [ ]:
import pandas as pd
import plotly.express as px

trials_df = study.trials_dataframe()
completed = trials_df[trials_df["state"] == "COMPLETE"].copy()
completed = completed[completed["value"] > float('-inf')].sort_values("value", ascending=False)

param_cols = [c for c in completed.columns if c.startswith("params_")]
attr_cols  = [c for c in completed.columns if c.startswith("user_attrs_") and
              any(k in c for k in ["sharpe","pnl","profit_factor","accuracy","drawdown","positions"])]

print(f"Completed: {len(completed)} / {len(trials_df)}")
print(f"Positive score: {(completed['value'] > 0).sum()}")
print(f"\nTop 15 trials:")
print(completed[["value"] + param_cols + attr_cols].head(15).to_string())

In [ ]:
# Param importance
importances = optuna.importance.get_param_importances(study)
imp_df = pd.DataFrame({"param": list(importances.keys()), "importance": list(importances.values())})
imp_df = imp_df.sort_values("importance", ascending=True)

fig = px.bar(imp_df, x="importance", y="param", orientation="h",
             title="V3 Parameter Importance (composite score)",
             color="importance", color_continuous_scale="Viridis")
fig.update_layout(plot_bgcolor="rgba(0,0,0,0.85)", paper_bgcolor="rgba(0,0,0,0.85)",
                  font=dict(color="white"), showlegend=False)
fig.show()

In [ ]:
# Stability check — top decile param ranges
top_n = max(10, len(completed) // 10)
top = completed.head(top_n)
print(f"Top {top_n} trials — composite score range: {top['value'].min():.3f} – {top['value'].max():.3f}")
print(f"\nParam ranges in top {top_n}:")
for col in param_cols:
    name = col.replace("params_", "")
    print(f"  {name:<30} {top[col].min():.4f} – {top[col].max():.4f}  (median {top[col].median():.4f})")

## Validate on Locked Bull Window — ONE SHOT

Run ONLY after params are finalized from stability analysis above.
This is the one-shot gate. Do not re-optimize based on this result.

Compare against V3 Baseline (no optimization): Sharpe -0.12, PF 2.46, PnL +$83

In [ ]:
VAL_START = datetime.datetime(2024, 7, 1)
VAL_END   = datetime.datetime(2025, 1, 1)

# Use stable-region params, not just single best trial
best = study.best_trial
best_params = best.params
print("Using params from best trial:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"  (composite score: {best.value:.4f})")
print("\n⚠️  If best trial is at search boundary, use median of top decile instead.")

In [ ]:
val_config = E1CompressionBreakoutConfig(
    id="e1_v3_validation",
    connector_name="binance_perpetual",
    trading_pair="BTC-USDT",
    total_amount_quote=Decimal("1000"),
    max_executors_per_side=1,
    cooldown_time=300, leverage=1,
    atr_period=14,
    atr_compression_window=90,
    atr_compression_threshold=best_params["atr_compression_threshold"],
    range_period=best_params["range_period"],
    volume_period=20,
    volume_floor_multiplier=best_params["volume_floor_multiplier"],
    volume_spike_multiplier=2.0,
    risk_off_ema_period=20, risk_off_threshold=0.97,
    trade_direction="BOTH",
    short_trend_filter=True,
    short_ema_slope_filter=False,
    short_trend_ema_period=best_params["short_trend_ema_period"],
    short_volume_floor_multiplier=best_params["volume_floor_multiplier"],
    stop_loss=Decimal("0.015"),
    take_profit=Decimal("0.03"),
    time_limit=86400,
    trailing_stop=TrailingStop(
        activation_price=Decimal("0.015"),
        trailing_delta=Decimal("0.005"),
    ),
    take_profit_order_type=OrderType.MARKET,
)

bt_val = BacktestingEngine(load_cached_data=True)
val_result = await bt_val.run_backtesting(
    config=val_config,
    start=int(VAL_START.timestamp()),
    end=int(VAL_END.timestamp()),
    backtesting_resolution="1m",
    trade_cost=0.000375,
)

r = val_result.results
if not isinstance(r.get("close_types"), dict):
    r["close_types"] = {}

exs = val_result.executors
longs  = [float(e.net_pnl_quote) for e in exs if e.config.side == TradeType.BUY]
shorts = [float(e.net_pnl_quote) for e in exs if e.config.side != TradeType.BUY]

print("=" * 55)
print("V3 OPTIMIZED — VALIDATION RESULT (Bull Locked)")
print("=" * 55)
print(val_result.get_results_summary())
print(f"Trades:    {len(exs)} ({len(longs)}L / {len(shorts)}S)")
print(f"Long PnL:  ${sum(longs):+.2f}  ({100*sum(longs)/r['net_pnl_quote']:.0f}% of gross)")
print(f"Short PnL: ${sum(shorts):+.2f}  ({100*sum(shorts)/r['net_pnl_quote']:.0f}% of gross)")
print()
print("── V3 reference (unoptimized) ──────────────")
print("  Sharpe -0.12 | PF 2.46 | PnL +$83 | 15 trades (9L/6S)")
print()
print(f"── Optimized delta ─────────────────────────")
sharpe_delta = (r.get("sharpe_ratio") or 0) - (-0.12)
pnl_delta    = r.get("net_pnl_quote", 0) - 83.32
print(f"  Sharpe: {sharpe_delta:+.3f}")
print(f"  PnL:    {pnl_delta:+.2f}")
print()
if abs(sharpe_delta) > 0.5 or abs(pnl_delta) > 40:
    print("⚠️  Large delta — inspect for overfitting before proceeding.")
else:
    print("✓  Delta within expected range.")